# ABM demo

Interactive version of the inspection ABM. Set the inspection weights and inspector parameters
with the sliders, press **Initialize**, then **Step** or **Play** to run the model tick by tick.
The grid shows the FBOs coloured by compliance level and the inspectors with their sight range;
below it, the compliance distribution and the caught rate are updated after each tick.



In [ ]:
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from datetime import datetime
from matplotlib.ticker import MaxNLocator
from mesa import Agent, Model
from mesa.space import MultiGrid
from mesa.time import RandomActivation
from mesa.datacollection import DataCollector
from ml_inference import predict_inspection_result
from IPython.display import display, Markdown
from ipywidgets import HBox, VBox, IntSlider, FloatSlider, Button, Play, jslink, Output, Label, ToggleButton

## Settings

In [ ]:
INPUT_FILE = "init_ABM_dataset.xlsx"

GRID_W, GRID_H = 90, 55
TICK_DAYS = 90
SIM_START_DATE = datetime.today().date()
POP_SEED = 42   

## Load data

In [ ]:
data = pd.read_excel(INPUT_FILE, parse_dates=["Last Inspection date"])


def load_rows_with_date(df, n, sim_start_date, sample_seed=POP_SEED):
    items = []
    for idx, r in df.sample(n, random_state=sample_seed).iterrows():
        last_date = r["Last Inspection date"].date()
        items.append({
            "agent_id": f"FBO_{idx}",
            "FBO type": r["FBO type"],
            "Product type": r["Product type"],
            "Operation type": r["Operation type"],
            "FBO Size": r["FBO Size"],
            "inspection_result": r["Inspection result"],
            "inspection_interval_days": max(0, (sim_start_date - last_date).days),
        })
    return items

## ABM

In [ ]:
# FBO agents
class FBOAgent(Agent):
    def __init__(self, uid, model, info):
        super().__init__(uid, model)
        self.agent_id = info["agent_id"]
        self.FBO_type = info["FBO type"]
        self.Product_type = info["Product type"]
        self.Operation_type = info["Operation type"]
        self.FBO_size = info["FBO Size"]
        self.inspection_result = info["inspection_result"]

        self.compliance_level = self.inspection_result
        self.actual_compliance_level = self.compliance_level   # latent continuous score
        self.former_inspection_result = self.inspection_result
        self.inspection_interval_days = info["inspection_interval_days"]

    def features(self):
        return {
            "FBO type": self.FBO_type,
            "Product type": self.Product_type,
            "Operation type": self.Operation_type,
            "Former Inspection result": self.former_inspection_result,
            "FBO Size": self.FBO_size,
            "Inspection interval (days)": self.inspection_interval_days,
        }

    def pre_ml_tick_update(self):
        # possible deterioration, for FBOs not inspected in the most recent tick
        ch = predict_inspection_result(self.features())
        if ch == 1 and self.inspection_interval_days != TICK_DAYS:
            self.actual_compliance_level = min(4, self.actual_compliance_level + self.model.deterioration_step)
            self.compliance_level = round(self.actual_compliance_level)

    def inspect_and_maybe_improve(self):
        self.inspection_result = self.compliance_level
        if self.compliance_level > 1:
            self.model.at_risk_inspected_this_tick += 1

        ch = predict_inspection_result(self.features())
        if ch == -1:
            self.compliance_level = max(1, self.compliance_level - self.model.improvement_step)

        self.actual_compliance_level = self.compliance_level
        self.inspection_interval_days = TICK_DAYS
        self.former_inspection_result = self.inspection_result


# Inspector agents
class InspectorAgent(Agent):
    def __init__(self, uid, model, move_range, sight, inspection_capacity):
        super().__init__(uid, model)
        self.move_range = move_range
        self.sight = sight
        self.inspection_capacity = inspection_capacity

    def step(self):
        # random move, try to land on an empty cell
        for _ in range(12):
            dx = random.randint(-self.move_range, self.move_range)
            dy = random.randint(-self.move_range, self.move_range)
            nx = (self.pos[0] + dx) % self.model.grid.width
            ny = (self.pos[1] + dy) % self.model.grid.height
            if self.model.grid.is_cell_empty((nx, ny)):
                break
        self.model.grid.move_agent(self, (nx, ny))

        nb = self.model.grid.get_neighbors(self.pos, moore=True, include_center=False, radius=self.sight)
        fbos = [a for a in nb if isinstance(a, FBOAgent)]
        if not fbos:
            return

        # rescale value of each factor to the same range
        def scale(arr):
            a = np.array(arr, dtype=float)
            mn, mx = a.min(), a.max()
            return np.full_like(a, 2.5) if mx == mn else 1 + 3 * (a - mn) / (mx - mn)

        sizes = scale([f.FBO_size for f in fbos])
        formers = scale([f.former_inspection_result for f in fbos])
        types = scale([f.FBO_type for f in fbos])
        products = scale([f.Product_type for f in fbos])
        intervals = scale([f.inspection_interval_days for f in fbos])

        w = self.model.weights
        score = (
            w["size"] * sizes
            + w["former"] * formers
            + w["type"] * types
            + w["product"] * products
            + w["interval"] * intervals
        )

        # all weights 0 is treated as random inspection
        random_condition = self.model.random_inspection or all(v == 0 for v in w.values())
        order = np.random.permutation(len(fbos)) if random_condition else np.argsort(-score)

        # Select FBOs to be inspected
        selected = []
        for idx in order:
            f = fbos[idx]
            if f not in self.model.assigned_this_tick:
                selected.append(f)
                self.model.assigned_this_tick.add(f)
            if len(selected) >= self.inspection_capacity:
                break

        for f in selected:
            f.inspect_and_maybe_improve()


# Model
class InspectionModel(Model):
    def __init__(
        self, items, num_inspectors=8, width=GRID_W, height=GRID_H,
        w_size=1.0, w_former=1.0, w_type=1.0, w_product=1.0, w_interval=1.0,
        move_range=18, sight=20, inspection_capacity=25, random_inspection=False,
        deterioration_step=0.10, improvement_step=1
    ):
        super().__init__()
        self.grid = MultiGrid(width, height, torus=True)
        self.schedule = RandomActivation(self)

        self.weights = {
            "size": w_size, "former": w_former, "type": w_type,
            "product": w_product, "interval": w_interval
        }
        self.random_inspection = random_inspection
        # Sensitivity parameters
        self.deterioration_step = deterioration_step
        self.improvement_step = improvement_step

        self.assigned_this_tick = set()
        self.at_risk_this_tick = 0
        self.at_risk_inspected_this_tick = 0

        # FBOs
        for info in items:
            f = FBOAgent(info["agent_id"], self, info)
            x, y = self.random.randrange(width), self.random.randrange(height)
            self.grid.place_agent(f, (x, y))
            self.schedule.add(f)

        # inspectors
        for i in range(num_inspectors):
            ins = InspectorAgent(f"INS_{i}", self, move_range, sight, inspection_capacity)
            while True:
                x, y = self.random.randrange(width), self.random.randrange(height)
                if self.grid.is_cell_empty((x, y)):
                    break
            self.grid.place_agent(ins, (x, y))
            self.schedule.add(ins)

        # DataCollector
        self.datacollector = DataCollector(model_reporters={
            "compliance_level_1": lambda m: sum(1 for a in m.schedule.agents
                                                if isinstance(a, FBOAgent) and a.compliance_level == 1),
            "compliance_level_2": lambda m: sum(1 for a in m.schedule.agents
                                                if isinstance(a, FBOAgent) and a.compliance_level == 2),
            "compliance_level_3": lambda m: sum(1 for a in m.schedule.agents
                                                if isinstance(a, FBOAgent) and a.compliance_level == 3),
            "compliance_level_4": lambda m: sum(1 for a in m.schedule.agents
                                                if isinstance(a, FBOAgent) and a.compliance_level == 4),

            "at_risk_start": lambda m: m.at_risk_this_tick,

            "caught_rate": lambda m: (
                0.0 if m.at_risk_this_tick == 0
                else 100.0 * m.at_risk_inspected_this_tick / m.at_risk_this_tick
            ),
        })

    def step(self):
        # Reset tick trackers
        self.assigned_this_tick.clear()
        self.at_risk_this_tick = 0
        self.at_risk_inspected_this_tick = 0

        # Pre-inspection update
        for a in self.schedule.agents:
            if isinstance(a, FBOAgent):
                a.pre_ml_tick_update()

        self.at_risk_this_tick = sum(
            1 for a in self.schedule.agents
            if isinstance(a, FBOAgent) and a.compliance_level > 1
        )

        # Inspectors
        for a in self.schedule.agents:
            if isinstance(a, InspectorAgent):
                a.step()

        # interval update for non-inspected
        for a in self.schedule.agents:
            if isinstance(a, FBOAgent) and a not in self.assigned_this_tick:
                a.inspection_interval_days += TICK_DAYS

        self.datacollector.collect(self)

## Grid

In [ ]:
LEVEL_COLORS = {1: "green", 2: "yellow", 3: "orange", 4: "red"}


def draw_human(ax, x, y, scale=0.8):
    ax.add_patch(plt.Circle((x, y - 0.2*scale), 0.2*scale, zorder=4))         # inspector head
    ax.plot([x, x], [y - 0.2*scale, y + 0.6*scale], zorder=4)                  # body
    ax.plot([x, x - 0.3*scale], [y + 0.2*scale, y + 0.4*scale], zorder=4)      # arms
    ax.plot([x, x + 0.3*scale], [y + 0.2*scale, y + 0.4*scale], zorder=4)
    ax.plot([x, x - 0.2*scale], [y + 0.6*scale, y + 1.0*scale], zorder=4)      # legs
    ax.plot([x, x + 0.2*scale], [y + 0.6*scale, y + 1.0*scale], zorder=4)


def snapshot_model(m):
    rows = []
    for a in m.schedule.agents:
        x, y = a.pos
        if isinstance(a, FBOAgent):
            rows.append({"kind": "FBO", "x": x, "y": y, "level": a.compliance_level,
                         "size": a.FBO_size, "type": a.FBO_type})
        else:
            rows.append({"kind": "inspector", "x": x, "y": y, "sight": a.sight})
    return pd.DataFrame(rows)


def plot_snapshot(snap):
    fig, ax = plt.subplots(figsize=(7, 7 * GRID_H / GRID_W))

    # FBOs: colour = compliance level, size = FBO size, shape = FBO type
    fbos = snap[snap["kind"] == "FBO"]
    type1 = fbos[fbos["type"] == 1]
    type2 = fbos[fbos["type"] == 2]
    ax.scatter(type1["x"], type1["y"], s=18 + 30*(type1["size"] - 1), c=type1["level"].map(LEVEL_COLORS),
               marker="o", alpha=0.9, zorder=2, label="FBO type 1")
    ax.scatter(type2["x"], type2["y"], s=18 + 30*(type2["size"] - 1), c=type2["level"].map(LEVEL_COLORS),
               marker="s", alpha=0.9, zorder=2, label="FBO type 2")

    # inspectors with their sight range
    inspectors = snap[snap["kind"] == "inspector"]
    for _, ins in inspectors.iterrows():
        draw_human(ax, ins["x"], ins["y"])
        ax.add_patch(plt.Circle((ins["x"], ins["y"]), ins["sight"], fill=False, zorder=3))

    ax.set_xlim(-0.5, GRID_W - 0.5)
    ax.set_ylim(-0.5, GRID_H - 0.5)
    ax.invert_yaxis()
    ax.set_aspect("equal", adjustable="box")
    ax.set_title("Grid (tick viewer)")
    ax.set_xlabel("x")
    ax.set_ylabel("y")
    ax.legend()
    plt.show()

## Slider and button

In [ ]:
w_random = ToggleButton(value=False, description="Random inspections",
                        tooltip="If on, inspectors choose FBOs randomly up to capacity")

w_w_size = FloatSlider(description="w_size", min=-4.0, max=4.0, step=0.1, value=0.0, readout_format=".1f")
w_w_former = FloatSlider(description="w_former", min=-4.0, max=4.0, step=0.1, value=1.5, readout_format=".1f")
w_w_type = FloatSlider(description="w_type", min=-4.0, max=4.0, step=0.1, value=-1.0, readout_format=".1f")
w_w_product = FloatSlider(description="w_product", min=-4.0, max=4.0, step=0.1, value=1.0, readout_format=".1f")
w_w_interval = FloatSlider(description="w_interval", min=-4.0, max=4.0, step=0.1, value=2.0, readout_format=".1f")

w_cap = IntSlider(description="capacity", min=1, max=50, step=1, value=25)
w_move = IntSlider(description="move_range", min=1, max=60, step=1, value=18)
w_sight = IntSlider(description="sight", min=1, max=60, step=1, value=20)
w_n_ins = IntSlider(description="inspectors", min=0, max=100, step=1, value=8)
w_n_fbo = IntSlider(description="N_FBO", min=500, max=5000, step=10, value=1000)
w_ticks = IntSlider(description="ticks", min=1, max=200, step=1, value=20)

btn_init = Button(description="Initialize")
btn_step = Button(description="Step")
play = Play(value=0, min=0, max=w_ticks.value, step=1, interval=600, description="Press play")
slider = IntSlider(description="frame", min=0, max=w_ticks.value, step=1, value=0)
jslink((play, "value"), (slider, "value"))  

out_grid = Output()
out_plots = Output()
out_info = Output()


def sync_tick_max(change):
    play.max = change["new"]
    slider.max = change["new"]

w_ticks.observe(sync_tick_max, names="value")

display(
    VBox([
        HBox([w_n_fbo, w_ticks, w_n_ins]),
        HBox([w_cap, w_move, w_sight]),
        HBox([w_w_size, w_w_former, w_w_type]),
        HBox([w_w_product, w_w_interval]),
        HBox([btn_init, btn_step, play, slider]),
        w_random,
        Label("Grid"),
        out_grid,
        Label("Plots & KPIs"),
        out_plots,
        Label("Info / Logs"),
        out_info,
    ])
)

with out_info:
    print("Click Initialize, then Step or Play.")

## Dashboard

In [ ]:
plt.rcParams.update({
    "figure.dpi": 160,
    "axes.labelsize": 9,
    "axes.titlesize": 10,
    "legend.fontsize": 9,
    "xtick.labelsize": 8,
    "ytick.labelsize": 8,
})

PLOT_COLORS = {
    "compliance_level_1": "#2ca02c",   # green
    "compliance_level_2": "#ffcc00",   # yellow
    "compliance_level_3": "#ff7f0e",   # orange
    "compliance_level_4": "#d62728",   # red
}

m = None          # current model
snaps = []        # one grid snapshot per tick, for the frame slider
cur_frame = 0     # current tick


def scenario_label():
    weights = [w_w_size.value, w_w_former.value, w_w_type.value, w_w_product.value, w_w_interval.value]
    if w_random.value or all(v == 0 for v in weights):
        return "(a) Random inspections"
    if len(set(weights)) == 1:
        return "(b) Equal-weight risk-based"
    return "(c) Adjusted-weight risk-based"


def show_grid(frame):
    with out_grid:
        out_grid.clear_output(wait=True)
        plot_snapshot(snaps[frame])


def show_results():
    df = m.datacollector.get_model_vars_dataframe().reset_index().rename(columns={"index": "tick"})

    with out_plots:
        out_plots.clear_output(wait=True)

        # caught rate per tick; the average excludes ticks without non-compliant FBOs
        kpis = df[["tick", "at_risk_start", "caught_rate"]].copy()
        kpis["avg_caught_rate"] = df.loc[df["at_risk_start"] > 0, "caught_rate"].mean()
        display(kpis.style.format({"caught_rate": "{:.1f}%", "avg_caught_rate": "{:.1f}%"})
                .set_caption("Caught rate (average excludes ticks with 0 non-compliant FBOs)"))

        # compliance distribution
        plt.figure(figsize=(5.5, 3.5))
        for col, color in PLOT_COLORS.items():
            plt.plot(df["tick"], df[col], label=col.replace("compliance_level_", "Level "),
                     linewidth=1.8, color=color)
        plt.title(f"Compliance distribution after inspections  {scenario_label()}")
        plt.xlabel("Tick")
        plt.ylabel("Number of FBOs")
        plt.gca().xaxis.set_major_locator(MaxNLocator(integer=True))
        plt.ylim(0, (w_n_fbo.value // 50 + 1) * 50)
        plt.legend(loc="upper center", bbox_to_anchor=(0.5, -0.18), ncol=4, frameon=False)
        plt.tight_layout(pad=1.5)
        plt.show()


@out_info.capture()
def build_model():
    global m, snaps, cur_frame
    items = load_rows_with_date(data, w_n_fbo.value, SIM_START_DATE)
    m = InspectionModel(
        items,
        num_inspectors=w_n_ins.value,
        w_size=w_w_size.value, w_former=w_w_former.value, w_type=w_w_type.value,
        w_product=w_w_product.value, w_interval=w_w_interval.value,
        move_range=w_move.value, sight=w_sight.value, inspection_capacity=w_cap.value,
        random_inspection=w_random.value,
    )
    snaps = [snapshot_model(m)]
    cur_frame = 0

    show_grid(0)
    out_plots.clear_output()
    out_info.clear_output(wait=True)
    print("Model initialized. Use Step or Play to advance ticks.")
    display(Markdown("**Tip:** Use the **Random inspections** switch *or* set all weights "
                     "(w_size, w_former, w_type, w_product, w_interval) to **0** to inspect randomly."))


@out_info.capture()
def step_once():
    global cur_frame
    if m is None:
        out_info.clear_output(wait=True)
        print("Model not initialized. Click Initialize first.")
        return
    if cur_frame >= w_ticks.value:
        return

    m.step()
    snaps.append(snapshot_model(m))
    cur_frame += 1

    show_grid(cur_frame)
    show_results()
    out_info.clear_output(wait=True)
    print(f"Tick {cur_frame}/{w_ticks.value} | Inspectors: {w_n_ins.value} | FBOs: {w_n_fbo.value}")


@out_info.capture()
def on_slider_change(change):
    if m is None:
        return
    target = change["new"]
    # run the model up to the chosen tick if it has not been run yet
    while len(snaps) - 1 < target and cur_frame < w_ticks.value:
        step_once()
    show_grid(target)
    show_results()


btn_init.on_click(lambda _: build_model())
btn_step.on_click(lambda _: step_once())
slider.observe(on_slider_change, names="value")